# VisionBridge A-Z base model training (error-resistant Colab)

Run these cells **from top to bottom**.

What this notebook does:
1. Creates an isolated **Python 3.12** environment with `uv`
2. Installs PyTorch (CUDA if Colab GPU is available), NumPy&lt;2, MediaPipe 0.10.35
3. Downloads the MediaPipe Hand Landmarker model
4. Downloads the RealSign Git-LFS dataset archive and extracts it
5. Extracts 126D two-hand landmarks (stratified 80/20 train/val from train+val pools)
6. Trains the letter base model (up to 500 epochs) and saves the best checkpoint

The original RealSign **test** split is left untouched and evaluated after training.

**Tips**
- Use a **GPU** runtime (Runtime → Change runtime type → GPU) for faster training.
- If a cell fails, read the printed stderr — this notebook surfaces subprocess output.
- Re-run from the top after a runtime restart.

In [ ]:
# Cell 1 — Environment setup (isolated Python 3.12 + deps)
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path("/content")
os.chdir(ROOT)
REPO = ROOT / "VisionBridge"
VENV = ROOT / "visionbridge_train_env"
PYTHON = VENV / "bin" / "python"


def run(cmd, env=None, check=True, cwd=None):
    """Run a command and always stream stdout/stderr so Colab shows the real error."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    result = subprocess.run(
        [str(c) for c in cmd],
        check=False,
        env=env,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"  {' '.join(str(c) for c in cmd)}\n"
            f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        )
    return result


for path in (REPO, VENV):
    if path.exists():
        print(f"Removing previous {path} ...")
        shutil.rmtree(path)

print("Cloning VisionBridge...")
run([
    "git", "clone", "--depth", "1",
    "https://github.com/BharathWaj-K-R/VisionBridge.git",
    str(REPO),
])

print("Installing uv...")
run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--disable-pip-version-check", "uv",
])

UV = [sys.executable, "-m", "uv"]

# Ensure a managed CPython 3.12 is available (Colab may ship 3.11/3.12/3.13)
print("Ensuring Python 3.12 is available via uv...")
run([*UV, "python", "install", "3.12"])

print("Creating isolated venv at", VENV)
run([*UV, "venv", "--python", "3.12", str(VENV)])

# Detect whether Colab has a GPU so we can install a matching torch wheel
gpu_probe = subprocess.run(
    ["nvidia-smi"], capture_output=True, text=True
)
has_gpu = gpu_probe.returncode == 0
print("GPU available (nvidia-smi):", has_gpu)

print("Installing torch, numpy<2, mediapipe==0.10.35 ...")
install_cmd = [
    *UV, "pip", "install", "--python", str(PYTHON),
    "numpy<2",
    "mediapipe==0.10.35",
    "opencv-python-headless",
    "Pillow",
]
if has_gpu:
    # Colab GPUs work with the default CUDA torch wheels from PyPI
    install_cmd.append("torch")
else:
    install_cmd.extend([
        "--extra-index-url", "https://download.pytorch.org/whl/cpu",
        "torch",
    ])

run(install_cmd)

# Colab sets MPLBACKEND=module://matplotlib_inline.backend_inline which is NOT
# installed inside the isolated venv. MediaPipe imports matplotlib on load, so
# force a headless backend for every venv subprocess.
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "backend")
env["MPLBACKEND"] = "Agg"
env.pop("MPLCONFIGDIR", None)

smoke = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
import sys
import torch
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

assert sys.version_info[:2] == (3, 12), sys.version_info
assert np.lib.NumpyVersion(np.__version__) < "2.0.0", np.__version__
assert mp.__version__ == "0.10.35", mp.__version__

print("MediaPipe Tasks API:", vision.HandLandmarker.__name__)
print("Training environment: PASS")
'''

run([str(PYTHON), "-c", smoke], env=env)
print("Cell 1 complete.")

In [ ]:
# Cell 2 — Download MediaPipe Hand Landmarker .task model
from pathlib import Path
import urllib.request

HAND_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
HAND_MODEL_PATH = Path("/content/hand_landmarker.task")

if HAND_MODEL_PATH.exists():
    HAND_MODEL_PATH.unlink()

print("Downloading MediaPipe Hand Landmarker model...")
urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL_PATH)

size = HAND_MODEL_PATH.stat().st_size
if size < 1_000_000:
    raise RuntimeError(f"Hand model download is unexpectedly small: {size} bytes")

print("Hand model bytes:", size)
print("Hand model download: PASS")

In [ ]:
# Cell 3 — Verify Hand Landmarker can load in the training venv
import os
import subprocess
import sys
from pathlib import Path

PYTHON = Path("/content/visionbridge_train_env/bin/python")
MODEL = Path("/content/hand_landmarker.task")
ENV = os.environ.copy()
ENV["MPLBACKEND"] = "Agg"

smoke = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = __import__("sys").argv[1]
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
)
detector = vision.HandLandmarker.create_from_options(options)
detector.close()
print("Hand Landmarker initialization: PASS")
'''

result = subprocess.run(
    [str(PYTHON), "-c", smoke, str(MODEL)],
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Hand Landmarker smoke test failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )

In [ ]:
# Cell 4 — Download and extract RealSign dataset (Git LFS media endpoint)
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

DATASET_URL = (
    "https://media.githubusercontent.com/media/RealSign62/"
    "RealSign-Indian-Sign-Language-Dataset/main/Dataset.zip"
)
ZIP_PATH = Path("/content/RealSign.zip")
DATASET_DIR = Path("/content/RealSign")

for path in (ZIP_PATH, DATASET_DIR):
    if path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

print("Downloading RealSign dataset (this can take a few minutes)...")
result = subprocess.run(
    [
        "curl", "-L", "--fail", "--retry", "5", "--retry-all-errors",
        "--progress-bar",
        "--output", str(ZIP_PATH),
        DATASET_URL,
    ],
    check=False,
    text=True,
    capture_output=True,
)
if result.stdout:
    print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"curl failed (exit {result.returncode}).\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}\n"
        "If this keeps failing, try Runtime → Restart runtime and re-run from Cell 1."
    )

if not ZIP_PATH.is_file() or ZIP_PATH.stat().st_size < 1_000_000:
    raise RuntimeError(
        f"Download looks incomplete: {ZIP_PATH} size={ZIP_PATH.stat().st_size if ZIP_PATH.exists() else 0}"
    )

if not zipfile.is_zipfile(ZIP_PATH):
    first_bytes = ZIP_PATH.read_bytes()[:200]
    raise RuntimeError(
        "RealSign download is not a valid ZIP archive. "
        "The Git LFS media request returned unexpected content: "
        + repr(first_bytes)
    )

print("RealSign archive bytes:", ZIP_PATH.stat().st_size)

print("Extracting...")
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(DATASET_DIR)

# Sanity-check that expected split folders exist somewhere under DATASET_DIR
expected_markers = ("Training", "Testing", "Validation")
found = {m: False for m in expected_markers}
for p in DATASET_DIR.rglob("*"):
    if p.is_dir():
        name = p.name
        for m in expected_markers:
            if m in name:
                found[m] = True
missing = [m for m, ok in found.items() if not ok]
if missing:
    print("WARNING: could not locate folders for:", missing)
    print("Top-level contents of", DATASET_DIR, ":")
    for child in sorted(DATASET_DIR.iterdir())[:30]:
        print(" ", child.name)
else:
    print("Found Training / Validation / Testing folders.")

print("RealSign extraction: PASS")

In [ ]:
# Cell 5 — Extract 126D landmarks (can take 10–40+ minutes depending on CPU)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

out_dir = Path("/content/visionbridge_letter_data")
if out_dir.exists():
    import shutil
    shutil.rmtree(out_dir)

command = [
    str(PYTHON),
    str(REPO / "backend/scripts/prepare_letter_dataset.py"),
    "--input-root", "/content/RealSign",
    "--output-dir", str(out_dir),
    "--validation-ratio", "0.20",
    "--seed", "42",
    "--hand-model-path", "/content/hand_landmarker.task",
]

print("Running landmark preparation (this is the slow step)...", flush=True)
result = subprocess.run(
    command,
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"prepare_letter_dataset.py failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
print("Landmark preparation: PASS")

In [ ]:
# Cell 6 — Validate prepared NPZ contract
from pathlib import Path
import json
import numpy as np

root = Path("/content/visionbridge_letter_data")
metadata = json.loads((root / "labels.json").read_text(encoding="utf-8"))

labels = metadata["labels"]
if labels != list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    raise RuntimeError(f"Expected A-Z labels, got {labels}")

print("Labels:", "".join(labels))
print("Split policy:", metadata["split_policy"])

for split in ("train", "val", "test"):
    data = np.load(root / f"{split}.npz")
    x = data["x"]
    y = data["y"]

    if x.ndim != 2 or x.shape[1] != 126:
        raise RuntimeError(f"{split} has invalid shape: {x.shape}")
    if y.ndim != 1 or len(x) != len(y) or len(x) == 0:
        raise RuntimeError(f"{split} has invalid labels")
    if not np.isfinite(x).all():
        raise RuntimeError(f"{split} contains NaN or Inf")

    counts = [int((y == i).sum()) for i in range(26)]
    if any(count == 0 for count in counts):
        raise RuntimeError(f"{split} is missing an A-Z class: {counts}")

    print(f"{split}: samples={len(x)} shape={x.shape}")

print("Prepared dataset contract: PASS")

In [ ]:
# Cell 7 — Train letter base model (up to 500 epochs; early-stops when all letters hit target)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

weights_dir = REPO / "backend/app/models/weights"
weights_dir.mkdir(parents=True, exist_ok=True)

command = [
    str(PYTHON),
    "-m",
    "app.training.letter_base",
    "--data-dir", "/content/visionbridge_letter_data",
    "--output", str(weights_dir / "letter_base_model.pt"),
    "--epochs", "500",
    "--batch-size", "128",
    "--lr", "0.001",
    "--weight-decay", "0.0001",
    "--target-class-accuracy", "1.0",
    "--seed", "42",
    "--hidden-dim", "128",
    "--embedding-dim", "64",
    "--dropout", "0.10",
]

print("Starting training...", flush=True)
result = subprocess.run(
    command,
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
# Training can print a lot; still surface full logs on failure
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Training failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
print("Training command finished successfully.")

In [ ]:
# Cell 8 — Verify checkpoint loads
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("Training did not create a checkpoint")

check = r'''
import sys
from app.models.letter_model import load_checkpoint

model = load_checkpoint(sys.argv[1])
print("CHECKPOINT LOAD: PASS")
print("input_dim =", model.input_dim)
print("hidden_dim =", model.hidden_dim)
print("embedding_dim =", model.embedding_dim)
print("classes =", model.num_classes)
print("labels =", "".join(model.labels))
'''

result = subprocess.run(
    [str(PYTHON), "-c", check, str(CHECKPOINT)],
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Checkpoint load failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )

print("Checkpoint bytes:", CHECKPOINT.stat().st_size)
print("Training pipeline: PASS")
print()
print("Download this file from Colab:")
print(" ", CHECKPOINT)

## Result

The final checkpoint is:

```
/content/VisionBridge/backend/app/models/weights/letter_base_model.pt
```

Copy or download that file into the same path in your local VisionBridge repository:

```
backend/app/models/weights/letter_base_model.pt
```

The notebook reports validation accuracy after every epoch and evaluates the untouched RealSign test split after training.

### If something still fails
1. **Runtime → Restart runtime**, then run all cells top-to-bottom.
2. Prefer a **GPU** runtime for training speed (landmark extraction is still mostly CPU).
3. Read the printed `stdout` / `stderr` from the failing cell — this version no longer hides subprocess errors.